# Прогнозирование конечных свойств композиционных материалов

**Автор:** Жихарев Вячеслав Сергеевич

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
matplotlib.style.use('seaborn-pastel')
np.set_printoptions(precision=3, suppress=True)

# Извлечение файлов, объединение таблиц и первичный просмотр данных

In [ ]:
# Опционально: скачать датасет напрямую из Google Drive
# Запусти эту ячейку один раз, если папки hw_data_composite еще нет
# !python3 -m pip install gdown
# !python3 download_data.py

In [ ]:
# загрузка данных из локальной папки с новым датасетом
def read_composite_part(path):
    data = pd.read_excel(path)
    if 'Unnamed: 0' in data.columns:
        return data.set_index('Unnamed: 0')

    unnamed_cols = [col for col in data.columns if str(col).startswith('Unnamed')]
    if unnamed_cols:
        data = data.drop(columns=unnamed_cols)

    return data

bp = read_composite_part('hw_data_composite/X_bp.xlsx')
nup = read_composite_part('hw_data_composite/X_nup.xlsx')

In [ ]:
bp

In [ ]:
nup

In [ ]:
# объединяем таблицы по индексу (INNER JOIN)
# оставляем только те индексы, которые есть в обеих таблицах
df = bp.join(nup, how='inner').reset_index(drop=True)

# сохраняем результат объединения в отдельный файл
merged_path = 'Dataset_composites_inner.csv'
df.to_csv(merged_path, index=False)
print(f'Объединенный датасет сохранен: {merged_path}, shape={df.shape}')

df

# Разведочный анализ данных

In [ ]:
df.info()  # проверяем типы данных

# проверяем наличие пропусков
missing = df.isna().sum().to_frame('missing_count')
missing['missing_percent'] = (missing['missing_count'] / len(df) * 100).round(2)
missing

In [ ]:
df.duplicated().sum() # проверяем наличие дубликатов


In [ ]:
# описательная статистика + среднее и медиана по каждому признаку
stats = df.describe().T.round(2)
stats['mean'] = df.mean(numeric_only=True).round(2)
stats['median'] = df.median(numeric_only=True).round(2)
stats

In [ ]:
# гистограммы распределения для каждой переменной
for col in df.columns:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[col], kde=True)
    plt.title(f'Распределение: {col}')
    plt.tight_layout()
    plt.show()

# попарные графики рассеяния
sns.pairplot(df)
plt.show()

In [ ]:
#строим матрицу корреляции
plt.figure(figsize=(15,6))
sns.heatmap(df.corr().round(2), mask = np.triu(df.corr()), annot=True, cmap='coolwarm')

## Общие первичные выводы

Распределение большинства признаков близко к нормальному.

Исключения:
- `Угол нашивки, град` (дискретный признак)
- `Поверхностная плотность, г/м2` (есть асимметрия)

Сильной корреляции между большей частью признаков не наблюдается.

## Детальный анализ признаков

In [ ]:
!pip install https://github.com/pandas-profiling/pandas-profiling/archive/master.zip
import pandas_profiling
df.profile_report()

In [ ]:
df.nunique() #Количество уникальных значений по каждому признаку

In [ ]:
# отрисовка ящиков с усами 
plt.figure(figsize=(25,10))
sns.boxplot(data=df, orient="h")
plt.xticks(rotation=45, ha='right')
plt.show()

# Подготовка данных для построения моделей

Выбросы не удаляем: значения выглядят как реальные технологические режимы, а не как случайные ошибки. Это помогает сохранить объем исходной выборки.

Удаляем строку с нулевыми значениями `Шаг нашивки` и `Плотность нашивки`: это, вероятнее всего, единичная ошибка в данных.

In [ ]:
# оцениваем количество выбросов по методу IQR
q1 = df.quantile(0.25)
q3 = df.quantile(0.75)
iqr = q3 - q1

outlier_mask = ((df < (q1 - 1.5 * iqr)) | (df > (q3 + 1.5 * iqr))).any(axis=1)
print('Количество выбросов по IQR:', outlier_mask.sum())

df.loc[outlier_mask].head()

In [ ]:
# удаляем выбросы по IQR и строку с нулевой плотностью нашивки
before_rows = len(df)

df = df[~outlier_mask].copy()
df = df[df['Плотность нашивки'] != 0].copy()

print('Удалено строк:', before_rows - len(df))
print('Новый размер данных:', df.shape)

## Поиск возможных искусственно сгенерированных наблюдений

Так как в датасете могут быть реальные и искусственно сгенерированные строки, добавим отдельную эвристику. Строка считается подозрительной, если она одновременно выбивается по нескольким статистическим методам.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

numeric = df.select_dtypes(include=[np.number]).copy()

q1 = numeric.quantile(0.25)
q3 = numeric.quantile(0.75)
iqr = q3 - q1
suspected_iqr = ((numeric < (q1 - 1.5 * iqr)) | (numeric > (q3 + 1.5 * iqr))).any(axis=1)

suspected_invalid = pd.Series(False, index=df.index)
for col in ['Шаг нашивки', 'Плотность нашивки']:
    if col in df.columns:
        suspected_invalid = suspected_invalid | (df[col] <= 0)

scaled = StandardScaler().fit_transform(numeric)
contamination = min(0.15, max(0.02, 10 / len(numeric)))

suspected_isolation = pd.Series(
    IsolationForest(contamination=contamination, random_state=42).fit_predict(scaled) == -1,
    index=df.index,
)

n_neighbors = min(20, len(numeric) - 1)
suspected_lof = pd.Series(
    LocalOutlierFactor(n_neighbors=n_neighbors, contamination=contamination).fit_predict(scaled) == -1,
    index=df.index,
)

votes = (
    suspected_iqr.astype(int)
    + suspected_isolation.astype(int)
    + suspected_lof.astype(int)
    + suspected_invalid.astype(int)
)

df_marked = df.copy()
df_marked['suspected_iqr'] = suspected_iqr
df_marked['suspected_isolation_forest'] = suspected_isolation
df_marked['suspected_lof'] = suspected_lof
df_marked['suspected_invalid_values'] = suspected_invalid
df_marked['suspected_votes'] = votes
df_marked['is_suspected_artificial'] = suspected_invalid | (votes >= 2)

df_clean = df_marked.loc[~df_marked['is_suspected_artificial'], df.columns].copy()

df_marked.to_csv('Dataset_composites_inner_marked.csv', index=False)
df_clean.to_csv('Dataset_composites_inner_clean.csv', index=False)

print('Всего строк:', len(df_marked))
print('Подозрительно искусственных строк:', int(df_marked['is_suspected_artificial'].sum()))
print('Строк после очистки:', len(df_clean))

df_marked['is_suspected_artificial'].value_counts()

## Нормализация данных

In [ ]:
#Нормализация данных
from sklearn.preprocessing import MinMaxScaler, StandardScaler, Normalizer

df_norm = MinMaxScaler().fit_transform(df)
df_norm = pd.DataFrame(data = df_norm, columns = df.columns)

df_st = StandardScaler().fit_transform(df)
df_st = pd.DataFrame(data = df_norm, columns = df.columns)

df_norm_n = Normalizer().fit_transform(df)
df_norm_n = pd.DataFrame(data = df_norm_n, columns = df.columns)

In [ ]:
df_norm

In [ ]:
df_st

In [ ]:
df_norm_n

In [ ]:
# отрисовка ящиков с усами 
plt.figure(figsize=(25,10))
sns.boxplot(data=df_norm, orient="h")
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
# отрисовка ящиков с усами 
plt.figure(figsize=(25,10))
sns.boxplot(data=df_st, orient="h")
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
# отрисовка ящиков с усами 
plt.figure(figsize=(25,10))
sns.boxplot(data=df_norm_n, orient="h")
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
df.plot(kind='kde', ax=ax)

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
df_norm.plot(kind='kde', ax=ax)

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
df_st.plot(kind='kde', ax=ax)

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
df_norm_n.plot(kind='kde', ax=ax)

Сохраняем итоговый датафрейм в `.csv` для следующих этапов работы.

In [ ]:
# сохраним датафрейм в csv для дальнейшей работы
df.to_csv('Dataset_composites.csv', index=False)

In [ ]:
!pip install xlwt
df.to_excel('Dataset_composites.xls', index=False)
df_norm.to_excel('Dataset_composites_norm.xls', index=False)